# 📦 Saudi Goods Root Category Classifier - AI GPU Engine (Google Colab)
This notebook runs the **Dense FAISS Vector Engine with High-Precision Multilingual AI Models** on Colab's GPU.

### 🌟 Supported Models:
1. **`BAAI/bge-m3`** *(Recommended)*: #1 World-class Multilingual Retrieval Model (1024-dim, 100% Free on GPU, No API key needed).
2. **`google-gemini`**: Google Gemini text-embedding-004 (768-dim, Free tier via Google AI Studio API key).
3. **`intfloat/multilingual-e5-large`**: 1024-dim large multilingual E5 model.
4. **`intfloat/multilingual-e5-small`**: 384-dim lightweight fast model.

### ⚡ Quick Steps:
1. Choose your model below (Default is **`BAAI/bge-m3`**).
2. Click the **Play (▶️)** button to build the FAISS index and start the server.
3. Copy the generated **Cloudflare Tunnel URL** (`https://xxxx.trycloudflare.com`).
4. Paste it into the **Colab AI Connection Bar** on Render: `https://saudi-goods-classifier.onrender.com`.

In [ ]:
# @title ⚙️ 1. Choose AI Model Engine / اختر محرك الذكاء الاصطناعي
MODEL_CHOICE = "BAAI/bge-m3" # @param ["BAAI/bge-m3", "google-gemini", "intfloat/multilingual-e5-large", "intfloat/multilingual-e5-small"]
GEMINI_API_KEY = "" # @param {type:"string"}
REBUILD_FAISS_INDEX = True # @param {type:"boolean"}

import os, subprocess, time, re, sys

# 1. Cleanly Clone or update repository
print("📥 Fetching latest code and Saudi market category contexts...")
%cd /content
if os.path.exists('/content/saudi-goods-classifier'):
    %cd /content/saudi-goods-classifier
    !git reset --hard HEAD
    !git pull origin main
else:
    !git clone https://github.com/Asad-allah/saudi-goods-classifier.git /content/saudi-goods-classifier
    %cd /content/saudi-goods-classifier

# 2. Install dependencies (PyTorch + FAISS + SentenceTransformers + Accelerate)
print("📦 Installing AI dependencies...")
!pip install -q --upgrade pip
!pip install -q -r requirements.txt sentence-transformers faiss-cpu accelerate

# 3. Build / Update FAISS Index directly in Colab on GPU for the chosen model
if REBUILD_FAISS_INDEX:
    print(f"\n🔨 Building High-Precision FAISS Index using model: {MODEL_CHOICE}...")
    if MODEL_CHOICE == "google-gemini":
        !PYTHONPATH=. python scripts/build_faiss_index.py --model google-gemini --gemini-api-key "$GEMINI_API_KEY"
    else:
        !PYTHONPATH=. python scripts/build_faiss_index.py --model "$MODEL_CHOICE"

# 4. Download cloudflared for free instant public tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 5. Launch FastAPI server with full semantic mode enabled
print(f"\n🚀 Starting Uvicorn API Server with GPU & Dense FAISS ({MODEL_CHOICE})...")
env = os.environ.copy()
env["PYTHONPATH"] = "."
env["DANDAN_ENABLE_SEMANTIC"] = "true"
env["DANDAN_DEMO_ENABLED"] = "true"
env["DANDAN_SEMANTIC_MODEL"] = MODEL_CHOICE
if GEMINI_API_KEY:
    env["GEMINI_API_KEY"] = GEMINI_API_KEY

server_proc = subprocess.Popen(["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"], env=env)
time.sleep(4)

print("🌐 Opening Cloudflare Public Tunnel...")
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

print("\n" + "=" * 78)
print("🎉 AI GPU SEMANTIC SERVER IS LIVE ON GOOGLE COLAB!")
print(f"🧠 Active Embedding Engine:  {MODEL_CHOICE}")
print(f"📌 Colab Public URL:         {tunnel_url}")
print(f"👉 Open your Render Dashboard: https://saudi-goods-classifier.onrender.com")
print(f"💡 Paste the URL above into the 'Colab AI' bar on Render to link GPU Semantic search!")
print("=" * 78 + "\n")

# Keep alive
server_proc.wait()
